In [30]:
import openai
import os
import json

from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, MatchValue

from langsmith import Client

In [5]:
qdrant_client = QdrantClient(url= "http://localhost:6333")

### Download data from Qdrant

In [8]:
all_points= qdrant_client.scroll(
    collection_name="Amazon-items-collection-00",
    limit=100,
    offset=None,
    with_payload=True,
    with_vectors=False
)

In [9]:
all_points

([Record(id=0, payload={'description': '2023 Upgraded 1080P WiFi Camera Motion Sound Detector for Home Office,Indoor Camera with Night Vision - Car Cameras for Surveillance - Size:1.22 x 1.22 x 1.22 inches ', 'image': 'https://m.media-amazon.com/images/I/41wwiQiqHPL._AC_.jpg', 'rating_number': 19, 'price': 18.85, 'average_rating': 2.1, 'parent_asin': 'B0BZYSHG74'}, vector=None, shard_key=None, order_value=None),
  Record(id=1, payload={'description': '65W Replacement AC Adapter Charger for HP 15 15-f233wm 15-f272wm 15-f010wm 15-f305dx 15-af131dx Spectre X360 13-a010dx 13-4003dx 13-4103dx Pavilion 15 Laptop Notebook PC Power Supply Cord FEATURES / POWER SPECS : Only Pwr+ Chargers Have Extra Long 12 Ft Power AC/DC cords / Output 19.5V 3.33A 45W 65W / Input 100-240V / Made in Taiwan COMPATIBILITY: HP 14 15 17 M6 M7 Series 14-dq0040nr 463958-001 741727-001 740015-001 740015-003 740015-004 Envy Stream 11 13 14 PRO G2 HP Spectre x360 13 13T Split x2, 612 G1, 410 G1, Elite X2 1011 G1, x360 31

In [11]:
all_points[0][0].payload

{'description': '2023 Upgraded 1080P WiFi Camera Motion Sound Detector for Home Office,Indoor Camera with Night Vision - Car Cameras for Surveillance - Size:1.22 x 1.22 x 1.22 inches ',
 'image': 'https://m.media-amazon.com/images/I/41wwiQiqHPL._AC_.jpg',
 'rating_number': 19,
 'price': 18.85,
 'average_rating': 2.1,
 'parent_asin': 'B0BZYSHG74'}

In [12]:
all_context = [{"id":data.payload["parent_asin"], "text": data.payload["description"]} for data in all_points[0]]

In [13]:
all_context

[{'id': 'B0BZYSHG74',
  'text': '2023 Upgraded 1080P WiFi Camera Motion Sound Detector for Home Office,Indoor Camera with Night Vision - Car Cameras for Surveillance - Size:1.22 x 1.22 x 1.22 inches '},
 {'id': 'B008FPY1VG',
  'text': '65W Replacement AC Adapter Charger for HP 15 15-f233wm 15-f272wm 15-f010wm 15-f305dx 15-af131dx Spectre X360 13-a010dx 13-4003dx 13-4103dx Pavilion 15 Laptop Notebook PC Power Supply Cord FEATURES / POWER SPECS : Only Pwr+ Chargers Have Extra Long 12 Ft Power AC/DC cords / Output 19.5V 3.33A 45W 65W / Input 100-240V / Made in Taiwan COMPATIBILITY: HP 14 15 17 M6 M7 Series 14-dq0040nr 463958-001 741727-001 740015-001 740015-003 740015-004 Envy Stream 11 13 14 PRO G2 HP Spectre x360 13 13T Split x2, 612 G1, 410 G1, Elite X2 1011 G1, x360 310 G2, ZBook 14 HP Chromebook 11 14 G1 G3 G4 Pavilion x360 14-dw1024nr; HP Notebook 15 17 Business Laptop 15-dy2024nr 15-af131dx 15-ay009dx 15-ay041wm 15-ba009dx 15-bs015dx 15-d035dx 15-f009wm 15-f010wm 15-f033wm 15-f039w

#### Render a prompt to generate synthetic Eval reference dataset

In [16]:
output_schema = {
    "type": "array",
    "items": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string", 
                "description": "Suggested question"
            },
            "chunk_ids": {
                "type": "array", 
                "items": {"type": "string", "description": "List of relevant chunk IDs from the retrieved context"
                }
            },
            "answer_example": {
                "type": "string",
                "description": "Suggested answer based on the retrieved context"
            },
            "reasoning": {
                "type": "string",
                "description": "Explanation of how the chunks supports the suggested answer"
            }
        }
    },
}


SYSTEM_PROMPT = f"""
I am building a RAG app. I have a collection of 50 chunks of text, each with a unique ID.
The RAG app will act as a shopping assistant thay can answer questions about products based on the chunks of text.
It will provide all the available products to you with ID's of each chunk. 
I want you to suggest 30 questions that a user might ask about the products to which the answers could be grounded in the chunks context.
The questions should be diverse and cover a wide range of topics related to the products, such as features, specifications, comparisons,
use cases, etc imitating a real potential user of this RAG system.
As an output, I want you to provide:
1. a list of questions, and IDs of the chunks that could be used to answer them.
2. provide an example answer to the question given the context of the chunks.
3. provide reasoning for why you think the chunks you selected are relevant to the question and answer.

Construct 10 that caould use multiple chunks to answer the question.
Construct 10 that could be answered using a single chunk.
Construct 5 that are more challenging and might require some reasoning or inference beyond just retrieving information from the chunks,
but still grounded in the information provided in the chunks.
Construct 5 that cannot be answered using the chunks to test the system's ability to recognize when it does not have enough information to answer a question.

<OUTPUT JSON SCHEMA>
{json.dumps(output_schema, indent=2)}
</OUTPUT JSON SCHEMA>

I need to be able to parse the json output.
"""

USER_PROMPT = f"""Here is the list of chunks, each list element is a dictionary with their IDs and text:
{all_context}
"""

In [17]:
print(SYSTEM_PROMPT)


I am building a RAG app. I have a collection of 50 chunks of text, each with a unique ID.
The RAG app will act as a shopping assistant thay can answer questions about products based on the chunks of text.
It will provide all the available products to you with ID's of each chunk. 
I want you to suggest 30 questions that a user might ask about the products to which the answers could be grounded in the chunks context.
The questions should be diverse and cover a wide range of topics related to the products, such as features, specifications, comparisons,
use cases, etc imitating a real potential user of this RAG system.
As an output, I want you to provide:
1. a list of questions, and IDs of the chunks that could be used to answer them.
2. provide an example answer to the question given the context of the chunks.
3. provide reasoning for why you think the chunks you selected are relevant to the question and answer.

Construct 10 that caould use multiple chunks to answer the question.
Constr

In [18]:
print(USER_PROMPT)

Here is the list of chunks, each list element is a dictionary with their IDs and text:
[{'id': 'B0BZYSHG74', 'text': '2023 Upgraded 1080P WiFi Camera Motion Sound Detector for Home Office,Indoor Camera with Night Vision - Car Cameras for Surveillance - Size:1.22 x 1.22 x 1.22 inches '}, {'id': 'B008FPY1VG', 'text': '65W Replacement AC Adapter Charger for HP 15 15-f233wm 15-f272wm 15-f010wm 15-f305dx 15-af131dx Spectre X360 13-a010dx 13-4003dx 13-4103dx Pavilion 15 Laptop Notebook PC Power Supply Cord FEATURES / POWER SPECS : Only Pwr+ Chargers Have Extra Long 12 Ft Power AC/DC cords / Output 19.5V 3.33A 45W 65W / Input 100-240V / Made in Taiwan COMPATIBILITY: HP 14 15 17 M6 M7 Series 14-dq0040nr 463958-001 741727-001 740015-001 740015-003 740015-004 Envy Stream 11 13 14 PRO G2 HP Spectre x360 13 13T Split x2, 612 G1, 410 G1, Elite X2 1011 G1, x360 310 G2, ZBook 14 HP Chromebook 11 14 G1 G3 G4 Pavilion x360 14-dw1024nr; HP Notebook 15 17 Business Laptop 15-dy2024nr 15-af131dx 15-ay009dx

In [19]:
response = openai.chat.completions.create(
    model="gpt-5-mini",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT}
    ],
    reasoning_effort="minimal",
    # temperature=0.7,
)

print(response.choices[0].message.content)

[
  {
    "question": "Does the 2023 Upgraded 1080P WiFi Camera support night vision and motion detection?",
    "chunk_ids": [
      "B0BZYSHG74"
    ],
    "answer_example": "Yes. The 2023 Upgraded 1080P WiFi Camera is described as a motion and sound detector indoor camera with night vision capabilities.",
    "reasoning": "The chunk B0BZYSHG74 explicitly names the product as a '1080P WiFi Camera Motion Sound Detector' and includes 'Night Vision' in its title, directly supporting both motion detection and night-vision features."
  },
  {
    "question": "What are the output specs of the 65W replacement AC adapter listed?",
    "chunk_ids": [
      "B008FPY1VG"
    ],
    "answer_example": "The adapter outputs 19.5V at 3.33A (45W) and is also described as 65W in the listing; overall it lists Output 19.5V 3.33A 45W / 65W depending on the variant.",
    "reasoning": "Chunk B008FPY1VG contains detailed power spec lines: 'Output 19.5V 3.33A 45W 65W', and extensive compatibility and safety

In [20]:
json_output = json.loads(response.choices[0].message.content)

In [21]:
len(json_output)

41

In [22]:
json_output

[{'question': 'Does the 2023 Upgraded 1080P WiFi Camera support night vision and motion detection?',
  'chunk_ids': ['B0BZYSHG74'],
  'answer_example': 'Yes. The 2023 Upgraded 1080P WiFi Camera is described as a motion and sound detector indoor camera with night vision capabilities.',
  'reasoning': "The chunk B0BZYSHG74 explicitly names the product as a '1080P WiFi Camera Motion Sound Detector' and includes 'Night Vision' in its title, directly supporting both motion detection and night-vision features."},
 {'question': 'What are the output specs of the 65W replacement AC adapter listed?',
  'chunk_ids': ['B008FPY1VG'],
  'answer_example': 'The adapter outputs 19.5V at 3.33A (45W) and is also described as 65W in the listing; overall it lists Output 19.5V 3.33A 45W / 65W depending on the variant.',
  'reasoning': "Chunk B008FPY1VG contains detailed power spec lines: 'Output 19.5V 3.33A 45W 65W', and extensive compatibility and safety/warranty info. That chunk is the single authoritativ

In [24]:
points = qdrant_client.scroll(
    collection_name="Amazon-items-collection-00",
    scroll_filter=Filter(
        must=[
            FieldCondition(
                key="parent_asin",
                match=MatchValue(value="B0C4P45FXV")
            )
        ]
    ),
    limit=100,
    with_payload=True,
    with_vectors=False
)[0]

In [25]:
points[0].payload

{'description': "YYJOY Spy Camera Hidden Camera Wireless Small Camera Nanny Cam,Wi-Fi,Night Vision,4K HD Mini Security Camera Indoor,Phone APP, Motion Detection,Battery Powered Spy Cam for Home,Easy to Use 𝐂𝐨𝐦𝐩𝐚𝐜𝐭 𝐚𝐧𝐝 𝐄𝐚𝐬𝐲 𝐭𝐨 𝐜𝐨𝐧𝐧𝐞𝐜𝐭: This camera is easy to install with magnetic base. The wifi connection is optimized for faster and more convenient scanning, and the app supports up to 4 cameras, making it perfect for home security or monitoring. 𝐖𝐢𝐫𝐞𝐥𝐞𝐬𝐬 𝐋𝐨𝐧𝐠-𝐥𝐚𝐬𝐭𝐢𝐧𝐠 𝐁𝐚𝐭𝐭𝐞𝐫𝐲 𝐋𝐢𝐟𝐞: Equipped with a 1200 mAh rechargeable battery, this wireless indoor camera can record continuously for 4 hours. You can also use the usb cable to powered by a power bank to seize the moment. 𝟒𝐊 𝐔𝐥𝐭𝐫𝐚 𝐇𝐃 𝐋𝐢𝐯𝐞 𝐕𝐢𝐝𝐞𝐨 & 𝐍𝐢𝐠𝐡𝐭 𝐕𝐢𝐬𝐢𝐨𝐧: Featuring a 160° wide-angle lens, this wireless security camera captures details with stunning 4K resolution. Night vision mode allows the camera to capture subjects up to 26 feet away in low light conditions. Also , this wireless camera is magnetic, easy to install in many places. 𝐒𝐦

In [27]:
def get_description(parent_asin:str)->str:
    points = qdrant_client.scroll(
    collection_name="Amazon-items-collection-00",
    scroll_filter=Filter(
        must=[
            FieldCondition(
                key="parent_asin",
                match=MatchValue(value=parent_asin)
            )
        ]
    ),
    limit=100,
    with_payload=True,
    with_vectors=False
)[0]

    return points[0].payload["description"]

In [28]:
get_description("B0C4P45FXV")

"YYJOY Spy Camera Hidden Camera Wireless Small Camera Nanny Cam,Wi-Fi,Night Vision,4K HD Mini Security Camera Indoor,Phone APP, Motion Detection,Battery Powered Spy Cam for Home,Easy to Use 𝐂𝐨𝐦𝐩𝐚𝐜𝐭 𝐚𝐧𝐝 𝐄𝐚𝐬𝐲 𝐭𝐨 𝐜𝐨𝐧𝐧𝐞𝐜𝐭: This camera is easy to install with magnetic base. The wifi connection is optimized for faster and more convenient scanning, and the app supports up to 4 cameras, making it perfect for home security or monitoring. 𝐖𝐢𝐫𝐞𝐥𝐞𝐬𝐬 𝐋𝐨𝐧𝐠-𝐥𝐚𝐬𝐭𝐢𝐧𝐠 𝐁𝐚𝐭𝐭𝐞𝐫𝐲 𝐋𝐢𝐟𝐞: Equipped with a 1200 mAh rechargeable battery, this wireless indoor camera can record continuously for 4 hours. You can also use the usb cable to powered by a power bank to seize the moment. 𝟒𝐊 𝐔𝐥𝐭𝐫𝐚 𝐇𝐃 𝐋𝐢𝐯𝐞 𝐕𝐢𝐝𝐞𝐨 & 𝐍𝐢𝐠𝐡𝐭 𝐕𝐢𝐬𝐢𝐨𝐧: Featuring a 160° wide-angle lens, this wireless security camera captures details with stunning 4K resolution. Night vision mode allows the camera to capture subjects up to 26 feet away in low light conditions. Also , this wireless camera is magnetic, easy to install in many places. 𝐒𝐦𝐚𝐫𝐭 𝐀𝐈 𝐇𝐮𝐦𝐚𝐧 𝐒𝐡𝐚

#### Create Eval dataset in Langsmith

In [31]:
client = Client(api_key=os.environ["LANGSMITH_API_KEY"])

In [32]:
dataset_name = "rag-evaluation-dataset"
dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Dataset for evaluating RAG pipeline"
)

In [ ]:
for item in json_output:
    client.create_example(
        dataset_id=dataset.id,
        inputs={"question": item["question"]},
        outputs={
            "ground_truth": item["answer_example"],
            "reference_context_ids": item["chunk_ids"],
            "reference_descriptions": [get_description(chunk_id) for chunk_id in item["chunk_ids"]]
        },
    )